# 19_final_training_data — 최종 학습데이터 구축 (1:1)

WEKA로 선택한 descriptor + ECFP4 fingerprint + potency + canonical SMILES 를 하나로 합쳐
최종 학습데이터를 만든다.

**구성:** `canonical_smiles` + `fingerprint(ECFP4 1024)` + `선택 descriptor` + `potency`
**저장:** CSV, Excel 각각

In [ ]:
# 프로젝트 루트로 이동 + import
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

FP_BITS = 1024   # ECFP4 (Morgan radius=2)
gen_ecfp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=FP_BITS)

### 입력 로드 (전체 descriptor Excel + WEKA 선택 목록)

In [ ]:
# 입력: (1) 전체 descriptor Excel(canonical_smiles+potency+217 desc)
#       (2) WEKA로 고른 descriptor 목록(filtered.csv 헤더)
FULL = 'data/HSD17B13_1to1_descriptors.xlsx'
FILT = 'data/HSD17B13_1to1_descriptors_weka_filtered.csv'

full = pd.read_excel(FULL)
sel_cols = [c for c in pd.read_csv(FILT, nrows=0).columns if c.lower() != 'potency']
print('WEKA 선택 descriptor:', len(sel_cols), '개')

miss = [c for c in sel_cols if c not in full.columns]
assert not miss, ('full에 없는 descriptor: %s' % miss)
print('전체 화합물:', len(full), '| potency 분포:', dict(full.potency.value_counts()))

### canonical SMILES → ECFP4 fingerprint(1024bit)

In [ ]:
# canonical SMILES -> ECFP4 fingerprint(1024 bit)
fp_rows, keep = [], []
for i, smi in enumerate(full['canonical_smiles']):
    m = Chem.MolFromSmiles(str(smi))
    if m is None:
        continue
    fp_rows.append(gen_ecfp.GetFingerprintAsNumPy(m))
    keep.append(i)
FP = pd.DataFrame(np.vstack(fp_rows).astype(np.int8),
                  columns=['fp_%04d' % j for j in range(FP_BITS)])
base = full.iloc[keep].reset_index(drop=True)
print('fingerprint 계산 완료:', FP.shape, '| 유효 화합물', len(base))

### 최종 조립 & 저장 (CSV + Excel)

In [ ]:
# 최종 조립: canonical_smiles + fingerprint(1024) + descriptor(36) + potency
final = pd.concat([
    base[['canonical_smiles']].reset_index(drop=True),   # 식별자
    FP.reset_index(drop=True),                           # fingerprint
    base[sel_cols].reset_index(drop=True),               # 선택 descriptor
    base[['potency']].reset_index(drop=True),            # 클래스(마지막)
], axis=1)
print('최종 학습데이터 shape:', final.shape,
      '(= canonical_smiles 1 + fp %d + desc %d + potency 1)' % (FP_BITS, len(sel_cols)))
print('구성 확인 → 첫 열:', final.columns[0], '| 마지막 열:', final.columns[-1])

OUT_CSV = 'data/HSD17B13_final_training_1to1.csv'
OUT_XLSX = 'data/HSD17B13_final_training_1to1.xlsx'
final.to_csv(OUT_CSV, index=False)
final.to_excel(OUT_XLSX, index=False)
print('저장 완료:')
print('  CSV :', OUT_CSV)
print('  XLSX:', OUT_XLSX)